# RAG Stress-Test — Retriever Ablation (Colab)

Runs `faiss_only`, `bm25_only`, and `no_rerank` ablations across all 8 perturbation conditions on a GPU.

**Before running this notebook:**
1. Push your latest code to GitHub: `git push origin main`
2. Upload `corpus.db`, `faiss.index`, `questions.json` to a folder named `rag_stress_test_data` in your Google Drive root.
3. **Runtime → Change runtime type → GPU (T4 or better)**
4. Then **Runtime → Run all** and wait. Total time ~1-2 hr on T4.

## 1. Mount Google Drive and verify data files exist

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA = '/content/drive/MyDrive/rag_stress_test_data'
for fname in ['corpus.db', 'faiss.index', 'questions.json']:
    p = os.path.join(DRIVE_DATA, fname)
    assert os.path.exists(p), f'MISSING: {p}'
    print(f'OK  {fname}: {os.path.getsize(p)/1e6:.1f} MB')

## 2. Clone the repo from GitHub

In [ ]:
%cd /content
!rm -rf rag_stress_test
!git clone https://github.com/Tanushree28/rag_stress_test.git
%cd rag_stress_test

## 3. Copy data files from Drive into the repo

In [ ]:
!mkdir -p data
!cp "/content/drive/MyDrive/rag_stress_test_data/corpus.db" data/
!cp "/content/drive/MyDrive/rag_stress_test_data/faiss.index" data/
!cp "/content/drive/MyDrive/rag_stress_test_data/questions.json" data/
!ls -lh data/

## 4. Install Python dependencies (~3 min)

In [ ]:
!pip install -q faiss-gpu-cu12 transformers sentence-transformers ollama scipy rouge-score numpy

## 5. Install + start Ollama, pull LLaMA 3.1:8b (~6 min one-time)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
# Start the ollama server in the background
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!ollama --version

In [ ]:
!ollama pull llama3.1:8b

## 6. Smoke test — make sure the pipeline runs at least once

In [ ]:
import sys; sys.path.insert(0, '.')
from retriever import retrieve
from generator import generate
passages = retrieve('What causes Hirschsprung disease?', top_k=3, mode='faiss_only')
print(f'Got {len(passages)} passages')
result = generate('What causes Hirschsprung disease?', passages)
print('Answer:', result['answer'][:200])

## 7. Run the ablation (~1-2 hrs on T4)

Watch the progress lines: `[mode] q X/Y -- done=N -- ETA M min`.

In [ ]:
!python run_ablation.py --n_per_type 25 --modes faiss_only,bm25_only,no_rerank

## 8. Save results back to Drive AND offer direct download

In [ ]:
import shutil, os
src = 'data/experiments_ablation.db'
assert os.path.exists(src), 'No results file -- did the run finish?'
shutil.copy(src, '/content/drive/MyDrive/rag_stress_test_data/experiments_ablation.db')
print('Copied to Drive: rag_stress_test_data/experiments_ablation.db')
print(f'Size: {os.path.getsize(src)/1e6:.2f} MB')
from google.colab import files
files.download(src)

## On your Mac, after download:

Move the downloaded `experiments_ablation.db` into your repo's `data/` folder, then merge:

```bash
sqlite3 data/experiments.db \
  "ATTACH 'data/experiments_ablation.db' AS abl; \
   INSERT INTO experiments \
     (timestamp, question_id, question_type, question_body, condition, \
      answer, metrics, passages, duration_s, retriever_mode) \
   SELECT timestamp, question_id, question_type, question_body, condition, \
          answer, metrics, passages, duration_s, retriever_mode \
   FROM abl.experiments;"
```

Then refresh `http://localhost:3000/analytics` and use the **Retriever** dropdown to compare modes.